In [2]:
import warnings

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        verbosity=0,
        early_stopping_rounds=50,
    )
    model.fit(
        train_df[feat_cols].values,
        np.log1p(train_df["target"].values),
        eval_set=[(val_df[feat_cols].values, np.log1p(val_df["target"].values))],
        verbose=False,
    )
    preds = np.expm1(model.predict(val_df[feat_cols].values))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val1_df = load_split("split2", feature_name)
    val2_df = load_split("split3", feature_name)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": "split1_" + feature_name, "groups": feature_name, "rmsle": score1})
    results.append({"experiment": "split2_" + feature_name, "groups": feature_name, "rmsle": score2})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val1_df = merge_splits("split2", subset)
    val2_df = merge_splits("split3", subset)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": f"split1_cumulative_{i}", "groups": label, "rmsle": score1})
    results.append({"experiment": f"split2_cumulative_{i}", "groups": label, "rmsle": score2})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


 split1 RMSLE = 1.74570  (best iteration: 152) | split2 RMSLE = 1.69897  (best iteration: 240)

Ranking by RMSLE (lower = better)
             experiment                                                                                 groups    rmsle
    split2_cumulative_8 base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.698972
    split2_cumulative_7       base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.698990
    split2_cumulative_6                          base + all_history + rolling_windows + velocity + trend + gmv 1.699149
    split2_cumulative_4                                        base + all_history + rolling_windows + velocity 1.699281
    split2_cumulative_3                                                   base + all_history + rolling_windows 1.699411
    split2_cumulative_5                                base + all_history + rolling_windows + velocity + trend 1.699428
        split2_velocity       

In [3]:
import joblib
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

RANDOM_STATE = 42

xgb_train = pd.read_parquet("datasets/split1_all_features.parquet")
xgb_val = pd.read_parquet("datasets/split2_all_features.parquet")
xgb_val2 = pd.read_parquet("datasets/split3_all_features.parquet")

xgb_feat_cols = [
    c for c in xgb_train.columns
    if c not in ("user_id", "target")
       and not pd.api.types.is_datetime64_any_dtype(xgb_train[c])
]

print(f"  XGBoost — train: {xgb_train.shape[0]} rows, {len(xgb_feat_cols)} features")

X_xgb_tr = xgb_train[xgb_feat_cols].values
y_xgb_tr = np.log1p(xgb_train["target"].values)
X_xgb_val = xgb_val[xgb_feat_cols].values
y_xgb_val = np.log1p(xgb_val["target"].values)
X_xgb_val2 = xgb_val2[xgb_feat_cols].values
y_xgb_val2 = np.log1p(xgb_val2["target"].values)
y_xgb_val_raw = xgb_val["target"].values
y_xgb_val2_raw = xgb_val2["target"].values

print("\nOptuna tuning XGBoost (20 trials)...")


def xgb_objective(trial):
    params = dict(
        objective="reg:squarederror",
        n_estimators=trial.suggest_int("n_estimators", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=50,
    )
    model = xgb.XGBRegressor(**params)
    model.fit(X_xgb_tr, y_xgb_tr, eval_set=[(X_xgb_val, y_xgb_val)], verbose=False)
    score1 = rmsle(y_xgb_val_raw, np.expm1(model.predict(X_xgb_val)))
    score2 = rmsle(y_xgb_val2_raw, np.expm1(model.predict(X_xgb_val2)))
    return (score1 + score2) / 2


xgb_study = optuna.create_study(direction="minimize")
xgb_study.optimize(xgb_objective, n_trials=20)
best_xgb_params = xgb_study.best_params

print(f"  Best XGBoost RMSLE: {xgb_study.best_value:.5f}")
print(f"  Params: {best_xgb_params}")

final_xgb = xgb.XGBRegressor(
    **best_xgb_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=50,
)
final_xgb.fit(X_xgb_tr, y_xgb_tr, eval_set=[(X_xgb_val, y_xgb_val)], verbose=False)

xgb_best_n = final_xgb.best_iteration + 1

X_xgb_full = np.concatenate([X_xgb_tr, X_xgb_val])
y_xgb_full = np.concatenate([y_xgb_tr, y_xgb_val])

xgb_sub_params = {k: v for k, v in best_xgb_params.items() if k not in ["n_estimators", "early_stopping_rounds"]}

final_xgb_full = xgb.XGBRegressor(
    **xgb_sub_params,
    n_estimators=xgb_best_n,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
final_xgb_full.fit(X_xgb_full, y_xgb_full)

joblib.dump(final_xgb, "model/xgb_split1.joblib")
joblib.dump(final_xgb_full, "model/xgb_full.joblib")

['model/xgb_full.joblib']

In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(feature_name: str, t: str) -> pd.DataFrame:
    path = f"datasets/{t}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(t: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(feature_names[0], t)
    for name in feature_names[1:]:
        extra = load_split(name, t)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    train_y = np.log1p(train_df["target"].values)
    val_y = np.log1p(val_df["target"].values)

    dtrain = xgb.DMatrix(train_df[feat_cols].values, label=train_y)
    dval = xgb.DMatrix(val_df[feat_cols].values, label=val_y)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "learning_rate": 0.05,
        "max_depth": 6,
        "verbosity": 0,
        "seed": 42,
    }

    evals_result = {}
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, "val")],
        early_stopping_rounds=50,
        evals_result=evals_result,
        verbose_eval=False,
    )

    preds = np.expm1(model.predict(dval))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split(feature_name, "train")
    val_df = load_split(feature_name, "test")
    score, best_iter = train_and_eval(train_df, val_df)
    print(f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": feature_name, "rmsle": score})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("train", subset)
    val_df = merge_splits("test", subset)
    score, best_iter = train_and_eval(train_df, val_df)
    print(f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": label, "rmsle": score})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


 RMSLE = 1.67169  (best iteration: 211)

Ranking by RMSLE (lower = better)
                                                                            experiment    rmsle
base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.671689
                                                  base + all_history + rolling_windows 1.671713
                         base + all_history + rolling_windows + velocity + trend + gmv 1.672239
      base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.672404
                                       base + all_history + rolling_windows + velocity 1.672470
                                                                                 trend 1.672582
                                                                       rolling_windows 1.672586
                               base + all_history + rolling_windows + velocity + trend 1.672594
                                                             

In [2]:
import joblib
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

RANDOM_STATE = 42

xgb_train = pd.read_parquet("datasets/train_all_features.parquet")
xgb_val = pd.read_parquet("datasets/test_all_features.parquet")

xgb_feat_cols = [
    c for c in xgb_train.columns
    if c not in ("user_id", "target")
       and not pd.api.types.is_datetime64_any_dtype(xgb_train[c])
]

print(f"  XGBoost — train: {xgb_train.shape[0]} rows, {len(xgb_feat_cols)} features")

X_xgb_tr = xgb_train[xgb_feat_cols].values
y_xgb_tr = np.log1p(xgb_train["target"].values)
X_xgb_val = xgb_val[xgb_feat_cols].values
y_xgb_val = np.log1p(xgb_val["target"].values)
y_xgb_val_raw = xgb_val["target"].values

print("\nOptuna tuning XGBoost (20 trials)...")


def xgb_objective(trial):
    params = dict(
        objective="reg:squarederror",
        n_estimators=trial.suggest_int("n_estimators", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=50,
    )
    model = xgb.XGBRegressor(**params)
    model.fit(X_xgb_tr, y_xgb_tr, eval_set=[(X_xgb_val, y_xgb_val)], verbose=False)
    score = rmsle(y_xgb_val_raw, np.expm1(model.predict(X_xgb_val)))
    return score


xgb_study = optuna.create_study(direction="minimize")
xgb_study.optimize(xgb_objective, n_trials=20)
best_xgb_params = xgb_study.best_params

print(f"  Best XGBoost RMSLE: {xgb_study.best_value:.5f}")
print(f"  Params: {best_xgb_params}")

final_xgb = xgb.XGBRegressor(
    **best_xgb_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=50,
)
final_xgb.fit(X_xgb_tr, y_xgb_tr, eval_set=[(X_xgb_val, y_xgb_val)], verbose=False)

xgb_best_n = final_xgb.best_iteration + 1

X_xgb_full = np.concatenate([X_xgb_tr, X_xgb_val])
y_xgb_full = np.concatenate([y_xgb_tr, y_xgb_val])

xgb_sub_params = {k: v for k, v in best_xgb_params.items() if k not in ["n_estimators", "early_stopping_rounds"]}

final_xgb_full = xgb.XGBRegressor(
    **xgb_sub_params,
    n_estimators=xgb_best_n,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
final_xgb_full.fit(X_xgb_full, y_xgb_full)

joblib.dump(final_xgb, "model/xgb_split1.joblib")
joblib.dump(final_xgb_full, "model/xgb_full.joblib")

['model/xgb_full.joblib']